<div style="background:linear-gradient(135deg,#1f4e79,#2f80ed 55%,#56ccf2);color:white;padding:26px 30px;border-radius:14px;margin:6px 0">
    <div style="font-size:2em;font-weight:800">
        Data Analysis and Plotting
    </div>
    <div style="font-size:1.1em;opacity:.95;margin-top:6px">
        Reading, slicing, grouping, plotting and testing one year of hourly PM2.5
    </div>
</div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    <div style="font-weight:700;font-size:1.08em;margin-bottom:4px">
        ✍️ How to use this notebook...
    </div>
    <ul>
        <li>
            <b>Shift + Enter</b> runs a cell. <b>Do not use “Run All.”</b> Run the cells in order from the top, because each one uses names made by the cells above it.
        </li>
        <li>
            Run the two setup cells first. They import the tools and set up the data. On Google Colab, remove the <code>#</code> in front of the install line first and run that cell once; the data downloads from GitHub, so you need an internet connection.
        </li>
        <li>
            Read each passage, then run the code under it and look at what comes out. Change a value and run it again.
        </li>
    </ul>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    <div style="font-weight:700;font-size:1.08em;margin-bottom:4px">
        🎯 Outline of this Course Module
    </div>
    <ul>
        <li>
            <b> Section 1: Importing the Test File </b>
        </li>
        <li>
            <b> Section 2: Slicing </b>
        </li>
        <li>
            <b> Section 3: Grouping </b>
        </li>
        <li>
            <b> Section 4: The Mean and Missing Data </b>
        </li>
        <li>
            <b> Section 5: Functions </b>
        </li>
        <li>
            <b> Section 6: Plotting with Matplotlib </b>
        </li>
        <li>
            <b> Section 7: Handling Time </b>
        </li>
        <li>
            <b> Section 8: SciPy </b>
        </li>
    </ul>
</div>

In [ ]:
# The tools used in this notebook.
import numpy as np                      # arrays and maths
import pandas as pd                     # tables (DataFrames); reading parquet files also needs pyarrow
                                        # (Google Colab already has it; elsewhere: pip install pyarrow)
import matplotlib.pyplot as plt         # plotting
import matplotlib.dates as mdates       # date axes
from scipy import stats                 # statistical tests and fits

In [ ]:
# OPTIONAL. Needed only if marama_2026 is not installed (Google Colab, for example).
# %pip install git+https://github.com/moroots/marama_2026

from marama_2026 import sample_datasets as sd    # small real datasets for this course

# The hourly file is read straight from its address in Section 1.
test_url = sd.BASE_URL + "session_2/AMP501_2420560-0.txt"
print(test_url)

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 1</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Importing the Test File</span>
    <div style="opacity:.92">
        Reading a pipe-separated AQS file into a clean table
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What does a data file look like before pandas has touched it? Open it as plain text and print three lines and the last one.
    <p>
        Line 0 begins with <code># RD</code> and holds the column names, separated by the pipe character <code>|</code> instead of commas. Line 1 begins with <code># RC</code> and is a second set of column names, for a kind of record that never appears in this file, so it has to be skipped. Line 2 is the first reading: monitor 0002, midnight on 1 January, 0.8. The last line is a footer that counts the records.
    </p>
</div>

In [ ]:
from urllib.request import urlopen

lines = urlopen(test_url).read().decode().splitlines()

for i in (0, 1, 2, -1):
    print(i, lines[i][:110])

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Why would I ask pandas to read nearly everything as text? Suppose I let it guess. The monitor ID <code>0002</code> looks like a number, so it arrives as <code>2.0</code>, and the leading zeros that name the monitor are gone. The date <code>20250101</code> becomes a number in the twenty millions, which is no use as a date.
    <p>
        So I pass <code>dtype=str</code>, skip line 1 with <code>skiprows=[1]</code>, and cut the footer off with <code>.iloc[:-1]</code>. Only the Sample Value column is truly a number, so that one I convert on purpose.
    </p>
</div>

In [ ]:
df = pd.read_csv(test_url, sep="|", skiprows=[1], dtype=str, low_memory=False).iloc[:-1]
df["Sample Value"] = pd.to_numeric(df["Sample Value"])

# Glue date and time into one timestamp (section 7 takes this line apart).
df["Datetime"] = pd.to_datetime(df["Date"] + df["Start Time"], format="%Y%m%d%H:%M")

print(df.shape)
df[["Site ID", "POC", "Datetime", "Sample Value"]].head(3)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How does a missing reading show up once the file is loaded? A blank in the Sample Value column became <b>NaN</b>, short for "not a number", which is how pandas marks a hole. Ask <code>isna()</code> and you get True where the hole is and False everywhere else. Python counts True as 1, so <code>sum()</code> counts the holes.
    <p>
        The file also has a Null Data Code column. Look at it only on the rows where the reading is blank, and it gives AQS's code for why.
    </p>
</div>

In [ ]:
blank = df["Sample Value"].isna()
print(blank.sum(), "blank hours out of", len(df))
df.loc[blank, "Null Data Code"].value_counts(dropna=False).head()

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 2</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Slicing</span>
    <div style="opacity:.92">
        Columns, rows, and the two ways to count them
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How do I keep only the columns I need? Square brackets with one name, <code>df["Sample Value"]</code>, give me that single column. Square brackets with a list of names, <code>df[["Site ID", "Datetime", "Sample Value"]]</code>, give me a smaller table.
    <p>
        The doubled brackets are not a new idea: the inner pair builds the list, and the outer pair does the selecting. The original table is untouched. I get a new one back, which I will call <code>hourly</code>, because each of its rows is one hour.
    </p>
</div>

In [ ]:
hourly = df[["Site ID", "POC", "Datetime", "Sample Value"]]
hourly.head(3)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How do I keep only the rows from one monitor? Start with the question. <code>hourly["Site ID"] == "0002"</code> asks it of every row at once and returns a column of True and False. Put that column inside the square brackets and pandas keeps the rows marked True.
    <p>
        To ask two questions, join them with <code>&amp;</code> and wrap each in parentheses. Python applies <code>&amp;</code> before it applies <code>==</code>, so without the parentheses it tries to combine the wrong things and fails.
    </p>
</div>

In [ ]:
avalon = hourly[hourly["Site ID"] == "0002"]
summer = avalon[(avalon["Datetime"] >= "2025-06-01") & (avalon["Datetime"] < "2025-09-01")]
print(len(avalon), len(summer))

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    If I want the first five rows, is it <code>0:5</code> or <code>0:4</code>? It depends on which door I use. <code>iloc</code> counts positions, and it follows the rule you know from lists: the stop is not included, so <code>iloc[0:5]</code> is rows 0 to 4. <code>loc</code> counts labels, the numbers written down the left side of the table, and a label range includes both ends, so <code>loc[0:4]</code> also returns five rows.
    <p>
        Here the labels happen to equal the positions. Filter the table and they stop matching: the rows keep their old labels while positions start again at 0, so the two doors point at different rows.
    </p>
</div>

In [ ]:
print(hourly.iloc[0:5].index.tolist(), hourly.loc[0:4].index.tolist())

liberty = hourly[hourly["Site ID"] == "0064"]
print(liberty.iloc[0:5].index.tolist(), liberty.loc[0:4].index.tolist())

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 3</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Grouping</span>
    <div style="opacity:.92">
        One answer per monitor, per month, per state
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How do I get one average for each monitor rather than one for the whole file? Picture the table dealt into eight piles, one per Site ID. Take the mean of Sample Value in each pile and lay the eight answers in a row.
    <p>
        That is <b>groupby</b>. You name the column that decides the piles, then the column to summarize, then what to do with it. Pandas leaves the blank hours out of each pile's mean.
    </p>
</div>

In [ ]:
hourly.groupby("Site ID")["Sample Value"].mean().round(2)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What if the pile should be a monitor in a month? Give <code>groupby</code> two keys instead of one, and the piles become pairs: monitor 0002 in January, monitor 0002 in February, and so on. The answer comes back with two levels of labels, which is awkward to read.
    <p>
        <code>unstack()</code> turns the inner level, the month, into columns, and you get a grid with monitors down the side and months across. Monitor 0067 has empty cells for its first nine months. Its pile was empty, and an empty pile has no mean.
    </p>
</div>

In [ ]:
month = hourly["Datetime"].dt.month        # 1 to 12, taken from the timestamp
hourly.groupby(["Site ID", month])["Sample Value"].mean().unstack().round(1)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What is inside EPA's pre-generated file, and why might one monitor-day appear twice? Suppose I ask for Avalon (state 42, county 003, site 0002) on 1 January 2025. I get two rows, not one. They share a POC, the number of the instrument at that site, and differ in Sample Duration: <code>1 HOUR</code> and <code>24-HR BLK AVG</code>.
    <p>
        The first is a mean of that day's hourly readings. The second is a 24-hour block average, and of the two only it carries an AQI value. A third kind, <code>24 HOUR</code>, comes from monitors that give one value a day. Counting rows by duration shows how many of each there are.
    </p>
</div>

In [ ]:
epa = sd.load_epa_daily_pm25()
epa["Site"] = epa["State Code"] + epa["County Code"] + epa["Site Num"]    # one ID for state + county + site

one_day = epa[(epa["Site"] == "420030002") & (epa["Date Local"] == "2025-01-01")]
print(one_day[["POC", "Sample Duration", "Observation Count", "Arithmetic Mean", "AQI"]])
print(epa["Sample Duration"].value_counts())

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Does one state have dirtier days than the others? To compare states I need one row per monitor-day, so I keep only the <code>24-HR BLK AVG</code> rows, the continuous monitors, and group by State Name. The median of the daily means is the middle day of each state's list, so a few smoky days cannot drag it around.
    <p>
        Named aggregation, <code>name=(column, function)</code>, lets me ask for three summaries at once. The count columns matter as much as the medians: West Virginia has five sites in this file and Pennsylvania has 52, so the medians are not equally well measured.
    </p>
</div>

In [ ]:
continuous = epa[epa["Sample Duration"] == "24-HR BLK AVG"]
by_state = continuous.groupby("State Name").agg(
    sites=("Site", "nunique"),
    site_days=("Arithmetic Mean", "count"),
    median_daily_mean=("Arithmetic Mean", "median"),
).round(1)
by_state.sort_values("median_daily_mean")

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Where did that parquet file come from, and could I fetch it myself? It is EPA's pre-generated daily PM2.5 file for 2025, trimmed to eight states. The optional cell below downloads the national file with the atmoz package and cuts it to the same states.
    <p>
        It is a download of several megabytes, and it calls a method whose name starts with an underscore, which by Python's convention means "internal". That is simply how the package exposes the download today. The result is a dictionary: the file name is the key and the table is the value. Skip the cell if you are offline.
    </p>
</div>

In [ ]:
# OPTIONAL. Needs:  pip install git+https://github.com/moroots/atmoz
try:
    from atmoz.surface.AirNow import epa_pregen
    # _download starts with an underscore (an internal method); it is how the package currently exposes this.
    out = epa_pregen._download(parameters="pm25", resolutions="daily", years=[2025])
    national = out["daily_88101_2025.csv"]
    same_states = national[national["State Name"].isin(epa["State Name"].unique())]
    print("national file:", national.shape, "  our eight states:", same_states.shape, "  parquet file:", epa.shape)
except ImportError:
    print("atmoz is not installed here. Skip this cell, or run:  pip install git+https://github.com/moroots/atmoz")
except Exception as err:
    print("The download did not work (offline?):", err)

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 4</span><span style="font-size:1.2em;font-weight:600"> &nbsp;The Mean and Missing Data</span>
    <div style="opacity:.92">
        What an average quietly leaves out
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What is the mean of 4, 6 and a missing value? NumPy refuses to guess: <code>np.mean([4, 6, np.nan])</code> returns <code>nan</code>, because anything added to a missing number is missing. Pandas makes the other choice and averages what is there, so the same three values in a <code>Series</code> give 5.0.
    <p>
        Both are defensible. Pandas's choice is the convenient one, and it is also the one that lets a day with three readings out of twenty-four report a perfectly confident mean. You can ask pandas for the strict behavior with <code>skipna=False</code>.
    </p>
</div>

In [ ]:
values = [4, 6, np.nan]
print(np.mean(values))
print(pd.Series(values).mean())
print(pd.Series(values).mean(skipna=False))

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Monitor 0067 has the lowest mean in the file, 5.73. Does it sit in the cleanest air? Look at when it was measuring: its first reading is on 9 October. The other monitors start on 1 January, except 1008, which starts on 9 May.
    <p>
        Cut all eight down to the same window, 9 October to 31 December, and 0067 is still the lowest, but the gap shrinks. The others now run from 6.37 to 7.53. A mean is the mean of the hours behind it, and those hours have to cover the same stretch of the year before two means can be compared.
    </p>
</div>

In [ ]:
window = hourly[hourly["Datetime"] >= "2025-10-09"]

summary = hourly.groupby("Site ID").agg(
    first_reading=("Datetime", "min"),
    valid_hours=("Sample Value", "count"),
    mean_all_hours=("Sample Value", "mean"),
)
summary["mean_since_9_oct"] = window.groupby("Site ID")["Sample Value"].mean()
summary.round({"mean_all_hours": 2, "mean_since_9_oct": 2})

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What is the daily mean of a day with only five hourly readings? Monitor 1376 on 1 May has exactly that, and their mean is 4.22. The arithmetic is fine, but five hours cannot speak for twenty-four.
    <p>
        So a daily mean needs a rule about how many valid hours the day must have. I will use 18 of 24, which is 75 percent. That is my choice, not a law of arithmetic. Later we will see that EPA's 24-hour block averages in this file begin at the same count.
    </p>
</div>

In [ ]:
day = hourly[(hourly["Site ID"] == "1376") & (hourly["Datetime"].dt.normalize() == "2025-05-01")]
print(day["Sample Value"].agg(["count", "mean"]))

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 5</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Functions</span>
    <div style="opacity:.92">
        Daily means with a completeness rule, checked against EPA
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    I want daily means, and I want to try different completeness rules. Do I retype the steps each time? No. They go in a function. The input is the hourly table and the minimum number of valid hours. The output has one row per monitor and day: the mean, and the count of valid hours behind it.
    <p>
        <code>min_hours=18</code> in the definition is a default. Call the function with only the table and you get 18; name the keyword and you get your own. Returning the count along with the mean lets anyone check the rule against the result.
    </p>
</div>

In [ ]:
def daily_means(table, min_hours=18):
    """One row per monitor, POC and day: the mean and the number of valid hours behind it."""
    work = table.assign(Day=table["Datetime"].dt.normalize())
    out = (work.groupby(["Site ID", "POC", "Day"])["Sample Value"]
               .agg(daily_mean="mean", valid_hours="count")
               .reset_index())
    return out[out["valid_hours"] >= min_hours]

daily = daily_means(hourly)
daily.head(3)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Does the 18-hour rule throw away much? Run the function three times and count the rows. With <code>min_hours=1</code>, any day with at least one reading survives. With 18, it is my rule. With 24, a day must be perfect.
    <p>
        The counts are 2,510, 2,493 and 2,426. Moving from 18 up to 24 costs 67 monitor-days, while moving down to 1 would add only 17.
    </p>
</div>

In [ ]:
for rule in (1, 18, 24):
    print(f"min_hours={rule:>2}:", len(daily_means(hourly, min_hours=rule)), "monitor-days")

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Do my daily means agree with EPA's? EPA's <code>1 HOUR</code> rows should be the same quantity, so I line them up by monitor, POC and date and subtract. A <b>merge</b> pastes two tables side by side wherever their key columns agree. I use <code>min_hours=1</code> so that I compare against every row EPA has, not only the days that pass my rule.
    <p>
        All 2,510 monitor-days find a partner. The largest difference is 4.8e-7, which is EPA rounding to six decimals, and EPA's Observation Count equals my valid hours on every row.
    </p>
</div>

In [ ]:
epa_hour = epa[(epa["Sample Duration"] == "1 HOUR") & (epa["Site"].str.startswith("42003"))]

mine = daily_means(hourly, min_hours=1)
mine = mine.assign(Site="42003" + mine["Site ID"], POC=mine["POC"].astype(int))

both = mine.merge(epa_hour, left_on=["Site", "POC", "Day"], right_on=["Site", "POC", "Date Local"])
diff = (both["daily_mean"] - both["Arithmetic Mean"]).abs()

print("mine:", len(mine), " EPA 1 HOUR rows:", len(epa_hour), " matched:", len(both))
print("largest difference:", diff.max())
print("hour counts equal on every row:", (both["valid_hours"] == both["Observation Count"]).all())

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Why do EPA's <code>24-HR BLK AVG</code> values not match mine exactly? Take Avalon on 1 January. My mean of its hours is 2.375, and EPA's value is 2.3. Rounding would give 2.4, so EPA is not rounding. It cuts my number off after the first decimal, which is called truncating.
    <p>
        EPA has such a row for 2,493 monitor-days, the same days that pass my 18-hour rule. EPA's value is never above my mean and never more than 0.1 below it, and truncating my means to one decimal reproduces EPA's on every row.
    </p>
</div>

In [ ]:
epa_blk = epa[(epa["Sample Duration"] == "24-HR BLK AVG") & (epa["Site"].str.startswith("42003"))]

mine18 = daily_means(hourly)                                   # the 18-hour rule
mine18 = mine18.assign(Site="42003" + mine18["Site ID"], POC=mine18["POC"].astype(int))
both = mine18.merge(epa_blk, left_on=["Site", "POC", "Day"], right_on=["Site", "POC", "Date Local"])

truncated = np.floor(both["daily_mean"] * 10 + 1e-9) / 10     # cut after one decimal (1e-9 guards against float noise)
gap = both["Arithmetic Mean"] - both["daily_mean"]

print("mine:", len(mine18), " EPA 24-HR BLK AVG rows:", len(epa_blk), " matched:", len(both))
print("EPA minus mine, smallest and largest:", gap.min().round(3), gap.max().round(3))
print("share equal to my mean truncated:", np.isclose(truncated, both["Arithmetic Mean"]).mean())

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 6</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Plotting with Matplotlib</span>
    <div style="opacity:.92">
        Figure, axes, labels, units and dates
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What are <code>fig</code> and <code>ax</code> in <code>fig, ax = plt.subplots()</code>? A matplotlib picture has two parts. The <b>figure</b> is the sheet of paper. The <b>axes</b> are the rectangle on it that holds the data, and the call hands me both. I draw on <code>ax</code>, and each <code>ax.plot(x, y)</code> adds one line.
    <p>
        Then I say what the numbers are, units included, since a number without a unit is only a drawing. Last I tell the date axis how to print: left alone, matplotlib picks something readable, but not always what I want.
    </p>
</div>

In [ ]:
wide = daily.pivot(index="Day", columns="Site ID", values="daily_mean")    # one column per monitor

fig, ax = plt.subplots(figsize=(10, 4), layout="tight")
ax.plot(wide.index, wide["0002"], label="Avalon (0002)", linewidth=1)
ax.plot(wide.index, wide["0064"], label="Liberty (0064)", linewidth=1)
ax.set_ylabel("PM2.5 daily mean (µg/m³)")
ax.set_xlabel("Date, 2025 (local standard time)")
ax.set_title("Daily mean PM2.5 at two Allegheny County monitors")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.grid(alpha=0.3)
ax.legend()
plt.show()

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How do I put eight monitors side by side in one picture? A bar chart answers one question: how big is each of these? Here it is the mean of each monitor's valid daily means.
    <p>
        The bars sit at the positions 0 to 7, and the names are labels I attach afterwards, which is why <code>ax.bar</code> takes two lists, where and how high. I write the number of days behind each bar on top of it, because the bar for 0067 rests on 83 days and most of the others on more than 350.
    </p>
</div>

In [ ]:
epa_ac = epa[epa["Site"].str.startswith("42003")]
names = epa_ac.drop_duplicates("Site Num").set_index("Site Num")["Local Site Name"]     # site number -> name

site_mean = daily.groupby("Site ID")["daily_mean"].agg(["mean", "count"])

fig, ax = plt.subplots(figsize=(9, 4), layout="tight")
bars = ax.bar(range(len(site_mean)), site_mean["mean"], color="tab:blue")
ax.bar_label(bars, labels=[f"{n} d" for n in site_mean["count"]])
ax.set_xticks(range(len(site_mean)), [f"{names[s]}\n({s})" for s in site_mean.index], rotation=30, ha="right")
ax.set_ylim(0, site_mean["mean"].max() * 1.15)      # headroom for the labels
ax.set_ylabel("Mean of valid daily means (µg/m³)")
ax.set_title("PM2.5 by monitor, 2025 (labels: days behind each bar)")
plt.show()

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    A bar shows an average, so how do I show the spread of the days as well? Take Avalon's 365 daily means and sort them. The box in a <b>box plot</b> runs from the 25th to the 75th place, so half the days fall inside it, and the line across the box is the median.
    <p>
        The whiskers reach to the farthest day within one and a half box-heights of the box. Every dot beyond a whisker is a day that far out, drawn individually.
    </p>
</div>

In [ ]:
order = list(wide.columns)

fig, ax = plt.subplots(figsize=(9, 4), layout="tight")
ax.boxplot([wide[s].dropna() for s in order])
ax.set_xticks(range(1, len(order) + 1), [names[s] for s in order], rotation=30, ha="right")
ax.set_ylabel("PM2.5 daily mean (µg/m³)")
ax.set_title("Spread of daily means by monitor, 2025")
ax.grid(axis="y", alpha=0.3)
plt.show()

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 7</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Handling Time</span>
    <div style="opacity:.92">
        Timestamps, time zones, resampling and the daily rhythm
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How did two text columns, <code>20250101</code> and <code>00:00</code>, become a moment in time? I glued them into <code>"2025010100:00"</code> and gave <code>pd.to_datetime</code> a template, <code>%Y%m%d%H:%M</code>. In the template <code>%Y</code> is a four-digit year, <code>%m</code> the month, <code>%d</code> the day, <code>%H</code> the hour, then a literal colon, then <code>%M</code> the minutes.
    <p>
        The result is a Timestamp. Its <code>.dt</code> attribute pulls out the parts: year, month, hour, day of the week. A reading stamped 07:00 is for the hour that starts at 07:00.
    </p>
</div>

In [ ]:
print(pd.to_datetime("2025010100:00", format="%Y%m%d%H:%M"))
print(hourly["Datetime"].dt.hour.head(3).tolist(), hourly["Datetime"].dt.day_name().iloc[0])

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What time is the file's 12:00 on a July clock in Pittsburgh? AQS stamps every reading in local standard time all year, with no daylight saving. So in July the file's 12:00 is 1 p.m. on the wall clock.
    <p>
        To ask Python, I attach a zone to the timestamp. Eastern standard time is five hours behind UTC, which the zone database writes as <code>Etc/GMT+5</code>; the sign looks backwards, and it is, on purpose. Then <code>tz_convert</code> restates the same instant in another zone. Attaching <code>America/New_York</code> at the start would be wrong, because it would apply daylight time to a stamp that never used it.
    </p>
</div>

In [ ]:
stamp = pd.Timestamp("2025-07-01 12:00").tz_localize("Etc/GMT+5")
print(stamp)
print(stamp.tz_convert("UTC"))
print(stamp.tz_convert("America/New_York"))

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How do I average by month without building a month column? Make the timestamps the index of the table, and <code>resample</code> does the grouping by time for me. <code>"MS"</code> means month start, so each bin is one calendar month, labeled with its first day. <code>"D"</code> would give days and <code>"W"</code> weeks.
    <p>
        The mean of each bin skips blank hours as before. It applies no completeness rule, so a month with only a few valid hours would still get a mean.
    </p>
</div>

In [ ]:
avalon.set_index("Datetime")["Sample Value"].resample("MS").mean().round(2)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Does Avalon have a daily rhythm, and is it the same in summer as in winter? Group by the hour of the day, 0 to 23, and average, once for June to August and once for December to February. The hours are standard time, so in summer the clock on the wall is an hour ahead of the horizontal axis.
    <p>
        The plot shows a curve for each season, and the cell prints where each one peaks.
    </p>
</div>

In [ ]:
hr = avalon["Datetime"].dt.hour
mo = avalon["Datetime"].dt.month
profile = {
    "Summer (Jun-Aug)": avalon[mo.isin([6, 7, 8])].groupby(hr)["Sample Value"].mean(),
    "Winter (Dec-Feb)": avalon[mo.isin([12, 1, 2])].groupby(hr)["Sample Value"].mean(),
}

fig, ax = plt.subplots(figsize=(8, 4), layout="tight")
for label, curve in profile.items():
    ax.plot(curve.index, curve.values, marker="o", markersize=3, label=label)
    print(f"{label}: peak at hour {curve.idxmax()} ({curve.max():.2f}), lowest at hour {curve.idxmin()} ({curve.min():.2f})")
ax.set_xticks(range(0, 24, 3))
ax.set_xlabel("Hour of day (local standard time, start of hour)")
ax.set_ylabel("Mean PM2.5 (µg/m³)")
ax.set_title("Avalon: average day by season, 2025")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 8</span><span style="font-size:1.2em;font-weight:600"> &nbsp;SciPy</span>
    <div style="opacity:.92">
        Skewness, a fitted line, a paired test and a percentile
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Avalon's daily means average 8.13, but their middle value, the median, is 6.91. Why do the two disagree? <code>scipy.stats.describe</code> gives the count, range, mean, variance and skewness in one call.
    <p>
        The skewness is 1.62. Positive skewness means a long tail on the right: a few high days pull the mean up, while the median only counts how many days lie on each side. The highest day here is 28.43, against a median of 6.91.
    </p>
</div>

In [ ]:
avalon_daily = wide["0002"].dropna()
print(stats.describe(avalon_daily))
print("median:", round(avalon_daily.median(), 2))

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    On days when Avalon is high, is Liberty high too? Pair up the 365 days on which both monitors have a daily mean. <code>linregress</code> fits the straight line through those pairs: Liberty equals slope times Avalon plus intercept.
    <p>
        The slope is 0.99 and the intercept is 0.89, so the line runs almost parallel to Liberty = Avalon, about 0.9 µg/m³ above it. The correlation r is 0.89, and r squared, 0.79, is the share of Liberty's day-to-day variation that the line accounts for.
    </p>
</div>

In [ ]:
pairs = wide[["0002", "0064"]].dropna()
fit = stats.linregress(pairs["0002"], pairs["0064"])
print(len(pairs), "paired days")
print(f"slope {fit.slope:.2f}, intercept {fit.intercept:.2f}, r {fit.rvalue:.2f}, r squared {fit.rvalue**2:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5), layout="tight")
ax.scatter(pairs["0002"], pairs["0064"], s=12, alpha=0.5)
xs = np.array([0, pairs["0002"].max()])
ax.plot(xs, fit.intercept + fit.slope * xs, color="darkred", label="fitted line")
ax.plot(xs, xs, color="gray", linestyle="--", label="1:1")
ax.set_xlabel("Avalon daily mean PM2.5 (µg/m³)")
ax.set_ylabel("Liberty daily mean PM2.5 (µg/m³)")
ax.set_title("Same days, two monitors")
ax.legend()
plt.show()

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Is Liberty really higher than Avalon, or is that chance? Across the 365 paired days Liberty is higher on 58 percent of them, its median advantage is 0.13 and its mean advantage 0.79. A p-value answers a narrow question: if the two monitors had no systematic difference, how often would the pairs look this lopsided?
    <p>
        <code>wilcoxon</code> compares each day's pair and returns about 1e-6, so almost never. That is evidence the difference is not zero. It does not say the difference matters. And it treats each day as an independent draw, which consecutive days of weather are not, so the printed p is probably too small.
    </p>
</div>

In [ ]:
gap = pairs["0064"] - pairs["0002"]
test = stats.wilcoxon(pairs["0064"], pairs["0002"])
print(f"Liberty higher on {(gap > 0).mean():.0%} of days; median gap {gap.median():.2f}, mean gap {gap.mean():.2f}")
print(f"Wilcoxon signed-rank p-value: {test.pvalue:.1e}")

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How often does Avalon stay under a given level? Take 12 µg/m³, a round number I picked, not a limit. Count the days below it and divide by the total days: that is <code>percentileofscore</code>, with the answer as a percentage.
    <p>
        With <code>kind="strict"</code> it counts only days strictly below 12. The result is 83.29, which is 304 of the 365 days.
    </p>
</div>

In [ ]:
pct = stats.percentileofscore(avalon_daily, 12, kind="strict")
print(round(pct, 2), "percent;", int((avalon_daily < 12).sum()), "of", len(avalon_daily), "days")

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    <div style="font-weight:700;font-size:1.08em;margin-bottom:4px">
        🎯 The vocabulary, at a glance
    </div>
    <ul>
        <li>
            <b>Reading a file:</b> <code>sep="|"</code>, <code>skiprows</code> and <code>dtype=str</code> describe the file; <code>.iloc[:-1]</code> drops a footer.
        </li>
        <li>
            <b>Slicing:</b> <code>df[["a", "b"]]</code> picks columns, <code>df[df["a"] == x]</code> picks rows, <code>iloc</code> counts positions (stop excluded), <code>loc</code> counts labels (both ends included).
        </li>
        <li>
            <b>Grouping:</b> <code>groupby(keys)[column].mean()</code>, <code>unstack()</code> for a grid, <code>agg(name=(column, function))</code> for several summaries.
        </li>
        <li>
            <b>Missing data:</b> NaN is skipped by pandas and spreads through NumPy. A mean is only as good as the hours behind it, so count them.
        </li>
        <li>
            <b>Functions:</b> a default such as <code>min_hours=18</code> can be overridden by keyword. <code>merge</code> joins two tables on key columns.
        </li>
        <li>
            <b>Matplotlib:</b> <code>fig, ax = plt.subplots()</code>, then draw on <code>ax</code>; always label axes with units.
        </li>
        <li>
            <b>Time:</b> <code>pd.to_datetime(text, format=...)</code>, <code>.dt</code> for parts, <code>tz_localize</code> / <code>tz_convert</code> for zones, <code>resample</code> for time bins. AQS time is local standard time.
        </li>
        <li>
            <b>SciPy:</b> <code>stats.describe</code>, <code>linregress</code>, <code>wilcoxon</code>, <code>percentileofscore</code>. A small p-value means the difference is probably not zero, not that it is large.
        </li>
    </ul>
</div>